# Fine-tune LFM2-700M on Aaron Rohrbacher Q&A → GGUF

End-to-end: train → merge → convert to GGUF → quantize → download.

**Requirements:** Colab with T4 GPU (Runtime → Change runtime type → T4 GPU). Free tier works.

**Time:** ~60-90 minutes for ~440 examples.

**Outputs:** `LFM2-700M-Q8_0-aaron.gguf` (~790 MB) downloaded to your local machine at the end.


## 1. Install dependencies

Unsloth handles LFM2's hybrid conv+attention architecture automatically and cuts VRAM ~50%.

In [ ]:
# Unsloth + deps. Remove %%capture if present so you can watch progress.
!pip install --upgrade pip
!pip install unsloth
# Pinned transformers/trl/peft for reproducibility
!pip install --upgrade 'transformers>=4.56' 'trl>=0.12' 'peft>=0.13' 'datasets>=3.0' 'bitsandbytes>=0.44' 'accelerate>=1.0'

## 2. Upload training data

Upload these five files from `training/` in the repo:
- `dataset_v2.jsonl` (~326 background Q&A — resume-derived)
- `dataset_v3_gap.jsonl` (~60 Q&A from your own answers in `gap_questions.md`: job-search logistics, tech preferences, experience depth, speaking, crisis stories)
- `dataset_v3_projects.jsonl` (~46 Q&A about portfolio projects from `src/info/Info.jsx`: Session, Heard, Fanboy, AppNow, MOVE, Klear, Thinger, ai, Nuel API, the site itself)
- `dataset_v3_resume.jsonl` (~34 verbatim resume sections from `src/components/resume/Resume.jsx`: each EXPERIENCE entry, each SKILL_GROUP, About text, full-resume overviews, meta-anchor examples)
- `dataset_v3_adversarial.jsonl` (~117 adversarial: declines, redirects, fabrication-bait, pronoun follow-ups, no-document-upload, multi-employer distinction, connect/escalation)

In [ ]:
from google.colab import files
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

## 3. Load LFM2-700M (4-bit) via Unsloth

Unsloth's pre-quantized build loads faster and uses ~50% less VRAM than stock.

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 1024  # Plenty for short Q&A; training examples are well under this.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/LFM2-700M-unsloth-bnb-4bit',
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=torch.bfloat16,
)

print('Model:', model.config._name_or_path if hasattr(model.config, '_name_or_path') else type(model).__name__)
print('Chat template present:', tokenizer.chat_template is not None)

## 4. Attach LoRA adapters

Unsloth auto-detects LFM2's attention projection modules and targets them.
Convolution blocks aren't LoRA-targetable by default — that's fine for instruction tuning.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                                   # LoRA rank
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'out_proj',
        'in_proj', 'w1', 'w2', 'w3',
    ],
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
model.print_trainable_parameters()

## 5. Load + format dataset

We prepend a system prompt to every example. The model learns to follow it at inference.

**If you change the system prompt here, change it in `src/components/ChatAgent.jsx` too** — keep training and inference identical.

In [ ]:
import json
from datasets import Dataset

SYSTEM_PROMPT = (
    "You are A-A-Bot, a chat assistant on Aaron Rohrbacher's portfolio site. "
    "You know Aaron's professional background, skills, and projects.\n\n"
    "Follow these rules in order, every turn:\n\n"
    "1. READ the facts below before answering. Your answer must be supported by an explicit statement in the facts.\n"
    "2. DO NOT infer, calculate, estimate, or combine facts to produce new claims. If the facts say \"January 2018\" but don't say \"8 years of experience,\" you do NOT compute the difference — you quote what's there or decline.\n"
    "3. DO NOT fabricate. If a name, date, number, employer, project, or detail is not literally written in the facts, you do not know it. Making up plausible-sounding information is the worst thing you can do.\n"
    "4. If the facts don't answer the question, say: \"I don't have that info — just say 'connect me' and I'll open a live chat with Aaron!\"\n"
    "5. For off-topic questions (other people, philosophy, politics, current events, weather, math), briefly redirect back to Aaron.\n"
    "6. Never ask the user to upload, provide, or share any document — you already know Aaron's background.\n"
    "7. Answer briefly — one or two sentences."
)

def load_jsonl(path):
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

background = load_jsonl('dataset_v2.jsonl')
gap = load_jsonl('dataset_v3_gap.jsonl')
projects = load_jsonl('dataset_v3_projects.jsonl')
resume = load_jsonl('dataset_v3_resume.jsonl')
adversarial = load_jsonl('dataset_v3_adversarial.jsonl')
all_records = background + gap + projects + resume + adversarial
print(f'Total examples: {len(all_records)} '
      f'({len(background)} bg + {len(gap)} gap + {len(projects)} projects + '
      f'{len(resume)} resume + {len(adversarial)} adversarial)')

def format_example(ex):
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}] + ex['messages']
    return {'text': tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

formatted = [format_example(ex) for ex in all_records]
dataset = Dataset.from_list(formatted)

# Shuffle once for training order; don't double the dataset (3 epochs already covers it).
train_dataset = dataset.shuffle(seed=42)
print(f'Training set: {len(train_dataset)} examples')
print('Sample formatted text:\n', train_dataset[0]['text'][:600])

## 6. Train

3 epochs, batch 2 with grad accum 4 (effective batch size 8), linear LR schedule with 5% warmup.
Expect ~30-60 minutes on T4 for ~880 examples (packing off).

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir='./outputs',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='linear',
    warmup_ratio=0.05,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_strategy='epoch',
    max_length=MAX_SEQ_LENGTH,      # renamed from max_seq_length in newer TRL
    packing=False,
    dataset_text_field='text',
    optim='adamw_8bit',
    report_to='none',
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    args=training_args,
)

trainer.train()

## 7. Quick sanity check

Run a couple of queries before conversion to make sure training worked.

In [ ]:
# Sanity check the trained model before merging. We intentionally do NOT call
# FastLanguageModel.for_inference(model) here — it mutates in-memory state in
# ways that can break the merge step below. Generation is slightly slower
# (~20-30s for six probes instead of ~5s) but merge stays reliable.

def ask(question):
    msgs = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': question},
    ]
    inputs = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors='pt',
    ).to('cuda')
    out = model.generate(
        inputs, max_new_tokens=120, do_sample=True, temperature=0.7, top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )
    text = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f'Q: {question}\nA: {text}\n')

ask('What does Aaron do?')
ask('What AWS services does he know?')
ask('How many brothers does Aaron have?')
ask('What is the meaning of life?')
ask('What is Aaron\'s wife\'s name?')
ask('Can you read his resume?')

## 8. Merge LoRA into base model

GGUF conversion needs a standalone model, not a LoRA adapter. Unsloth's `save_pretrained_merged` handles dequantization + merge in one step.

In [ ]:
import os, torch

MERGED_DIR = '/content/lfm2-700m-aaron-merged'

# Primary: Unsloth's save_pretrained_merged handles 4-bit → fp16 dequant,
# LoRA merge, and save in one call. Absolute path avoids cwd issues.
# Fallback: manual merge_and_unload with save_original_format=False to
# sidestep the NotImplementedError some transformers versions throw when
# trying to reverse 4-bit weight conversions during save.

def merged_looks_valid(path):
    return (
        os.path.isdir(path)
        and os.path.isfile(os.path.join(path, 'config.json'))
        and any(f.endswith('.safetensors') or f.endswith('.bin')
                for f in os.listdir(path))
    )

primary_err = None
try:
    print('[merge] primary path: Unsloth save_pretrained_merged(merged_16bit)')
    model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method='merged_16bit')
except Exception as e:
    primary_err = e
    print('[merge] primary failed:', type(e).__name__, '-', str(e)[:200])

if not merged_looks_valid(MERGED_DIR):
    print('[merge] falling back to manual merge_and_unload + save_pretrained(save_original_format=False)')
    try:
        merged = model.merge_and_unload()
    except Exception as e:
        print('[merge] merge_and_unload failed:', type(e).__name__, '-', str(e)[:200])
        raise
    try:
        merged.save_pretrained(MERGED_DIR, safe_serialization=True, save_original_format=False)
    except TypeError:
        # Older transformers doesn't accept save_original_format — try without.
        merged.save_pretrained(MERGED_DIR, safe_serialization=True)
    tokenizer.save_pretrained(MERGED_DIR)

if not merged_looks_valid(MERGED_DIR):
    raise RuntimeError(
        f'Both merge paths failed. Primary error: {primary_err}. '
        f'Check {MERGED_DIR} contents.'
    )

print('[merge] OK — merged model at', MERGED_DIR)
!ls -la {MERGED_DIR}

## 9. Clone and build llama.cpp

We need `convert_hf_to_gguf.py` (has native LFM2 support) and `llama-quantize`.

In [ ]:
# Clone + build llama.cpp (just the llama-quantize target). ~2-5 min.
# Progress will scroll as it runs — git clone → pip install → cmake configure → compile.
!git clone --depth=1 https://github.com/ggml-org/llama.cpp.git /content/llama.cpp
!pip install -r /content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
# Build just llama-quantize (not the full project — saves 5+ min)
!cmake -B /content/llama.cpp/build /content/llama.cpp -DGGML_CUDA=OFF -DLLAMA_BUILD_SERVER=OFF -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF
!cmake --build /content/llama.cpp/build --target llama-quantize -j $(nproc)

## 10. Convert merged model → GGUF (F16)

In [ ]:
F16_PATH = '/content/lfm2-700m-aaron-f16.gguf'
# Absolute path — robust regardless of current working directory.
!python /content/llama.cpp/convert_hf_to_gguf.py /content/lfm2-700m-aaron-merged --outfile {F16_PATH} --outtype f16
!ls -lh {F16_PATH}

## 11. Quantize F16 → Q8_0

Q8_0 is what the site currently loads (`LFM2-700M-Q8_0.gguf`). Near-FP16 quality, ~50% smaller file.

In [ ]:
Q8_PATH = '/content/LFM2-700M-Q8_0-aaron.gguf'
!/content/llama.cpp/build/bin/llama-quantize {F16_PATH} {Q8_PATH} Q8_0
!ls -lh {Q8_PATH}

## 12. Download the quantized GGUF

This triggers a browser download. The file lands in your Downloads folder.

**Next steps (local):**
1. Move the downloaded file to `public/models/lfm2-700m-gguf/LFM2-700M-Q8_0-aaron.gguf` in the repo
2. Update `src/components/ChatAgent.jsx` to point at the new filename
3. Update the system prompt in `ChatAgent.jsx` to match `SYSTEM_PROMPT` from step 5

In [ ]:
files.download(Q8_PATH)